# Phase 4 (Part A): Material/Vendor Deep-Dive

This notebook queries the `material_vendor` table from our database and generates a **Vendor Scorecard** calculating:
- On-time delivery rate (%)
- Average wastage (%)
- Number of orders fulfilled
- **Average delay days** and **delay_days_std** (delivery timing variability)

We rank the vendors based on these reliability metrics and export the final scorecard to `reports/vendor_scorecard.csv`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src folder to path
sys.path.append(os.path.abspath('../src'))

import db_connection

## 1. Load Material Vendor Data

In [ ]:
conn = db_connection.get_connection()
df_mv = pd.read_sql_query("SELECT * FROM material_vendor", conn)
conn.close()

df_mv.head()

## 2. Compute Vendor Metrics & Generate Scorecard

Group by `vendor_name` and `material_type` to calculate:
- `total_orders`: count of orders
- `on_time_delivery_rate`: percentage of orders delivered on time
- `average_wastage_pct`: mean wastage percentage
- `avg_delay_days` / `delay_days_std`: average and standard deviation of delivery delay
  (actual - promised delivery date, in days). `delay_days_std` captures how *consistent*
  a vendor's delivery timing is, on top of the plain on-time rate.

In [ ]:
df_mv["promised_delivery_date"] = pd.to_datetime(df_mv["promised_delivery_date"])
df_mv["actual_delivery_date"] = pd.to_datetime(df_mv["actual_delivery_date"])
df_mv["delay_days"] = (df_mv["actual_delivery_date"] - df_mv["promised_delivery_date"]).dt.days

scorecard = df_mv.groupby(["vendor_name", "material_type"]).agg(
    total_orders=("record_id", "count"),
    on_time_orders=("on_time", "sum"),
    avg_wastage_pct=("wastage_pct", "mean"),
    total_ordered_qty=("ordered_qty", "sum"),
    total_delivered_qty=("delivered_qty", "sum"),
    avg_delay_days=("delay_days", "mean"),
    delay_days_std=("delay_days", "std"),
).reset_index()

scorecard["on_time_delivery_rate_pct"] = (scorecard["on_time_orders"] / scorecard["total_orders"]) * 100
scorecard["qty_fulfillment_rate_pct"] = (scorecard["total_delivered_qty"] / scorecard["total_ordered_qty"]) * 100

scorecard["avg_wastage_pct"] = scorecard["avg_wastage_pct"].round(2)
scorecard["on_time_delivery_rate_pct"] = scorecard["on_time_delivery_rate_pct"].round(2)
scorecard["qty_fulfillment_rate_pct"] = scorecard["qty_fulfillment_rate_pct"].round(2)
scorecard["avg_delay_days"] = scorecard["avg_delay_days"].round(2)
# delay_days_std is NaN when a vendor+material pair has only 1 order
scorecard["delay_days_std"] = scorecard["delay_days_std"].fillna(0).round(2)

scorecard = scorecard.sort_values(by=["on_time_delivery_rate_pct", "avg_wastage_pct"], ascending=[False, True])
scorecard

## 3. Export Scorecard

In [ ]:
os.makedirs("../reports", exist_ok=True)
scorecard.to_csv("../reports/vendor_scorecard.csv", index=False)
print("Vendor scorecard exported to reports/vendor_scorecard.csv")

## 4. Visualization of Vendor Performance

In [ ]:
sns.set_theme(style="whitegrid")

# Plot: On-Time Delivery Rate and Average Wastage by Vendor
plt.figure(figsize=(12, 6))
sns.scatterplot(
    data=scorecard,
    x="on_time_delivery_rate_pct",
    y="avg_wastage_pct",
    hue="material_type",
    size="total_orders",
    sizes=(40, 200),
    palette="deep"
)
for i, row in scorecard.iterrows():
    plt.text(
        row["on_time_delivery_rate_pct"] + 0.5,
        row["avg_wastage_pct"] - 0.1,
        row["vendor_name"],
        fontsize=9,
        alpha=0.8
    )

plt.title("Vendor Performance: On-Time Delivery vs. Wastage Rate", fontsize=14, fontweight='bold')
plt.xlabel("On-Time Delivery Rate (%)", fontsize=12)
plt.ylabel("Average Wastage %", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("../reports/vendor_performance_scatter.png", dpi=300)
plt.show()